In [1]:
import pandas as pd

In [ ]:
DATA_DIR = 'data' 
df = pd.read_parquet('hypoglycemia.parquet')

In [3]:
hypoglycemia_codes = ["LOINC/14749-6", "SNOMED/33747003", "LOINC/LP416145-3"]
# remove all df rows that have a code in hypoglycemia_codes
print(f"Before removing hypoglycemia codes: {len(df)} rows")
df = df[~df['code'].isin(hypoglycemia_codes)]
print(f"After removing hypoglycemia codes: {len(df)} rows")

Before removing hypoglycemia codes: 11964722 rows
After removing hypoglycemia codes: 11956950 rows


In [4]:
labels = pd.read_parquet('hypoglycemia_label2.parquet')
# rename 'value' to 'label'
labels.rename(columns={'value': 'label'}, inplace=True)
labels.head()

,patient_id,prediction_time,label,value_multi,start,Sample_time,Time
0,115967106,2023-01-09 10:03:00,True,3,2019-02-18 10:12:00,NaT,2023-01-09 10:03:00
1,115967181,2021-01-15 14:46:00,True,3,2021-01-09 23:59:00,NaT,2021-01-15 14:46:00
2,115967229,2021-10-04 07:22:00,True,3,2020-09-05 00:05:00,NaT,2021-10-04 07:22:00
3,115967241,2019-08-22 10:45:00,True,3,2019-08-17 23:59:00,NaT,2019-08-22 10:45:00
4,115967258,2006-11-21 06:29:00,True,3,2006-07-08 05:50:00,NaT,2006-11-21 06:29:00


In [5]:
# find positive fraction 
pos_labels = labels[labels['label'] == True]
print("Positive fraction: ", len(pos_labels) / len(labels))

Positive fraction:  0.05745393634840871


In [6]:
# find time difference between 'Time' and 'start' columns
labels['Time'] = pd.to_datetime(labels['Time'])
labels['visit_start'] = pd.to_datetime(labels['start'])
labels['prefix_start'] = labels['Time'] - pd.Timedelta(hours=4)
labels['visit_end'] = labels['Time'] - pd.Timedelta(hours=0.5)
labels.head()

,patient_id,prediction_time,label,value_multi,start,Sample_time,Time,visit_start,prefix_start,visit_end
0,115967106,2023-01-09 10:03:00,True,3,2019-02-18 10:12:00,NaT,2023-01-09 10:03:00,2019-02-18 10:12:00,2023-01-09 06:03:00,2023-01-09 09:33:00
1,115967181,2021-01-15 14:46:00,True,3,2021-01-09 23:59:00,NaT,2021-01-15 14:46:00,2021-01-09 23:59:00,2021-01-15 10:46:00,2021-01-15 14:16:00
2,115967229,2021-10-04 07:22:00,True,3,2020-09-05 00:05:00,NaT,2021-10-04 07:22:00,2020-09-05 00:05:00,2021-10-04 03:22:00,2021-10-04 06:52:00
3,115967241,2019-08-22 10:45:00,True,3,2019-08-17 23:59:00,NaT,2019-08-22 10:45:00,2019-08-17 23:59:00,2019-08-22 06:45:00,2019-08-22 10:15:00
4,115967258,2006-11-21 06:29:00,True,3,2006-07-08 05:50:00,NaT,2006-11-21 06:29:00,2006-07-08 05:50:00,2006-11-21 02:29:00,2006-11-21 05:59:00


In [7]:
# drop all columns in df except patient_id,start_time, code, value, 
df = df[['patient_id', 'start', 'code', 'value']]
# rename 'start' to 'event_time'
df.rename(columns={'start': 'event_time'}, inplace=True)
# drop all columns in labels except patient_id, visit_start, label, visit_end
labels = labels[['patient_id', 'visit_start', 'label', 'prefix_start', 'visit_end']]

In [8]:
# merge my_df and my_labels on patient_id
merged_df = df.merge(labels, on='patient_id', how='inner')
# filter my_df to only include rows where start_time is between start and end
merged_df = merged_df[(merged_df['event_time'] >= merged_df['visit_start']) & (merged_df['event_time'] <= merged_df['visit_end'])]

In [9]:
# add a new column 'in_prefix' that is True if event_time is after prefix_start
merged_df['in_prefix'] = merged_df['event_time'] >= merged_df['prefix_start']

In [10]:
# For every patient_id, find the rows where in_prefix is True
# and then of those rows, find the number of unique event_time values for each patient_id
# find the mean, mean of the number of unique event_time values for each patient_id

true_count = merged_df[merged_df['in_prefix'] == True].groupby('patient_id')['event_time'].nunique()
mean_true_count = true_count.mean()
print("Mean number of unique event_time values for each patient_id where in_prefix is True: ", mean_true_count)
print("Median number of unique event_time values for each patient_id where in_prefix is True: ", true_count.median())


Mean number of unique event_time values for each patient_id where in_prefix is True:  9.581395348837209
Median number of unique event_time values for each patient_id where in_prefix is True:  4.5


In [11]:
# keep only patient_id, event_time, code, value
merged_df = merged_df[['patient_id', 'event_time', 'code', 'value', 'in_prefix']]

In [12]:
# sort by patient_id, event_time
merged_df = merged_df.sort_values(by=['patient_id', 'event_time'])
# keep only the last 300 rows for each patient_id
# merged_df = merged_df.groupby('patient_id')

In [13]:
# for the `value` in merged_df, if the value can be interpreted as a float, convert it to float with 2 decimal places
def convert_to_float(value):
    try:
        return round(float(value), 2)
    except:
        return value
merged_df['value'] = merged_df['value'].apply(convert_to_float)

In [14]:
# compute the min event_time for each patient, and then create a new column `relative_time_mins` which is the difference between event_time and the min event_time for that patient
merged_df['event_time'] = pd.to_datetime(merged_df['event_time'])
merged_df['min_event_time'] = merged_df.groupby('patient_id')['event_time'].transform('min')
merged_df['relative_time_mins'] = (merged_df['event_time'] - merged_df['min_event_time']).dt.total_seconds() / 60.0
# drop the min_event_time column
merged_df.drop(columns=['min_event_time'], inplace=True)
merged_df

,patient_id,event_time,code,value,in_prefix,relative_time_mins
0,115967095,2008-10-07 23:40:00,LOINC/9279-1,16.0,False,0.0
1,115967095,2008-10-07 23:40:00,LOINC/8867-4,88.0,False,0.0
2,115967095,2008-10-07 23:40:00,LOINC/8310-5,98.6,False,0.0
3,115967095,2008-10-07 23:40:00,LOINC/8462-4,58.0,False,0.0
4,115967095,2008-10-07 23:40:00,LOINC/8327-9,1.0,False,0.0
...,...,...,...,...,...,...
11956026,121124990,2018-03-20 23:59:00,RxNorm/2267305,None,False,5760.0
11956027,121124990,2018-03-20 23:59:00,Domain/OMOP generated,None,False,5760.0
11956028,121124990,2018-03-20 23:59:00,Domain/OMOP generated,None,False,5760.0
11956029,121124990,2018-03-20 23:59:00,SNOMED/230056004,N,False,5760.0


In [ ]:
all_codes = pd.read_parquet('all_codes.parquet')

In [16]:
# join merged_df with all_codes on code
merged_df = merged_df.merge(all_codes, on='code', how='left')


In [17]:
# in 'description' column remove all occurrences of '#' or 'Bld Auto', or any text in parentheses eg '(foo)'
merged_df['description'] = merged_df['description'].str.replace('#', '', regex=False)
merged_df['description'] = merged_df['description'].str.replace('Bld Auto', '', regex=False)
merged_df['description'] = merged_df['description'].str.replace(r'\(.*?\)', '', regex=True)

In [18]:
# drop all nan values in the description column
merged_df = merged_df[merged_df['description'].notna()]
# strip all whitespace from the description column
merged_df['description'] = merged_df['description'].str.strip()

In [19]:
# create text column by concatenating the description and value columns ONLY if the value is not nan
# else, just use the description column
merged_df['Text'] = merged_df.apply(lambda x: x['description'] + ': ' + str(x['value']) if pd.notna(x['value']) else x['description'], axis=1)

In [50]:
# For every patient_id, find the rows where in_prefix is True
# and then of those rows, find the number of unique event_time values for each patient_id
# find the mean, mean of the number of unique event_time values for each patient_id
merged_df = merged_df.groupby('patient_id').tail(500)
true_count = merged_df[merged_df['in_prefix'] == True].groupby('patient_id')['event_time'].nunique()
mean_true_count = true_count.mean()
print("Mean number of unique event_time values for each patient_id where in_prefix is True: ", mean_true_count)
print("Median number of unique event_time values for each patient_id where in_prefix is True: ", true_count.median())
print("Number of patients with in_prefix: ", len(merged_df[merged_df['in_prefix'] == True]['patient_id'].unique()))

Mean number of unique event_time values for each patient_id where in_prefix is True:  7.74020618556701
Median number of unique event_time values for each patient_id where in_prefix is True:  4.0
Number of patients with in_prefix:  485


In [79]:
# group by patien_id 
def aggregate_same_times(group):
    group = group.groupby('relative_time_mins').agg({
        'patient_id': 'first',
        'Text': lambda x: ";".join(x.astype(str)),
        'in_prefix': 'sum',
    }).reset_index()
    group['in_prefix'] = group['in_prefix'].astype(bool)
    return group

merged_group = merged_df.groupby('patient_id').apply(aggregate_same_times).reset_index(drop=True)
merged_group['Text'] = merged_group['relative_time_mins'].astype(str) + " - " + merged_group['Text'] + "\n"


/tmp/ipykernel_2058407/155258434.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_group = merged_df.groupby('patient_id').apply(aggregate_same_times).reset_index(drop=True)


In [69]:
merged_group

,relative_time_mins,patient_id,Text,in_prefix
0,1933.0,115967095,1933.0 - BP dias: 61.0;Resp rate: 16.0;BP mean...,False
1,1935.0,115967095,1935.0 - insulin lispro 100 UNT/ML Injectable ...,False
2,1940.0,115967095,1940.0 - 2 ML ondansetron 2 MG/ML Injection\n,False
3,2017.0,115967095,2017.0 - Consult note\n,False
4,2074.0,115967095,2074.0 - Nutr+diet Consult note\n,False
...,...,...,...,...
226060,5541.0,121124990,5541.0 - Body temperature: 98.1;BP dias: 63.0;...,False
226061,5579.0,121124990,5579.0 - Heart rate: 73.0\n,False
226062,5580.0,121124990,5580.0 - Heart rate: 70.0\n,False
226063,5701.0,121124990,5701.0 - Body temperature: 97.9;BP dias: 75.0;...,False


In [80]:
# Now, group by patient_id, and compute the cumulative prefix of the Text column; each row should be the cumulative of the 
def compute_cumulative_prefixes(group):
    group = group.sort_values(by='relative_time_mins')
    group['Text'] = group['Text'].cumsum()
    return group
merged_group = merged_group.groupby('patient_id').apply(compute_cumulative_prefixes).reset_index(drop=True)
merged_group

/tmp/ipykernel_2058407/1030687554.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  merged_group = merged_group.groupby('patient_id').apply(compute_cumulative_prefixes).reset_index(drop=True)


,relative_time_mins,patient_id,Text,in_prefix
0,1933.0,115967095,1933.0 - BP dias: 61.0;Resp rate: 16.0;BP mean...,False
1,1935.0,115967095,1933.0 - BP dias: 61.0;Resp rate: 16.0;BP mean...,False
2,1940.0,115967095,1933.0 - BP dias: 61.0;Resp rate: 16.0;BP mean...,False
3,2017.0,115967095,1933.0 - BP dias: 61.0;Resp rate: 16.0;BP mean...,False
4,2074.0,115967095,1933.0 - BP dias: 61.0;Resp rate: 16.0;BP mean...,False
...,...,...,...,...
226060,5541.0,121124990,0.0 - Pipe tobacco consumption: N;High risk pr...,False
226061,5579.0,121124990,0.0 - Pipe tobacco consumption: N;High risk pr...,False
226062,5580.0,121124990,0.0 - Pipe tobacco consumption: N;High risk pr...,False
226063,5701.0,121124990,0.0 - Pipe tobacco consumption: N;High risk pr...,False


In [82]:
merged_group = merged_group[merged_group['in_prefix'] == True]

In [ ]:
# merge merged_group with labels on patient_id
final = merged_group.merge(labels, on='patient_id', how='inner')
# drop all columns except patient_id, relative_time_mins, Text, label
final = final[['patient_id', 'relative_time_mins', 'Text', 'label']]


,patient_id,relative_time_mins,Text,label
0,115967106,2046048.0,2043768.0 - BP mean: 75.0;BP sys: 108.0;Q-T in...,True
1,115967106,2046088.0,2043768.0 - BP mean: 75.0;BP sys: 108.0;Q-T in...,True
2,115967106,2046093.0,2043768.0 - BP mean: 75.0;BP sys: 108.0;Q-T in...,True
3,115967106,2046108.0,2043768.0 - BP mean: 75.0;BP sys: 108.0;Q-T in...,True
4,115967106,2046109.0,2043768.0 - BP mean: 75.0;BP sys: 108.0;Q-T in...,True
...,...,...,...,...
3749,115973818,2255.0,0.0 - BMI: 26.84;Primary malignant neoplasm of...,False
3750,115973818,2257.0,0.0 - BMI: 26.84;Primary malignant neoplasm of...,False
3751,115973818,2290.0,0.0 - BMI: 26.84;Primary malignant neoplasm of...,False
3752,115973818,2296.0,0.0 - BMI: 26.84;Primary malignant neoplasm of...,False


In [85]:
# rename label to Label, relative_time_mins to TimeUpto
final.rename(columns={'label': 'Label', 'relative_time_mins': 'TimeUpto'}, inplace=True)

In [86]:
# compute the positive fraction
pos_labels = final[final['Label'] == True]
print("Positive fraction: ", len(pos_labels) / len(final))

Positive fraction:  0.6859350026638252


In [87]:
# save final to hypoglycemia_stability.parquet
final.to_parquet('hypoglycemia_stability.parquet', index=False)

In [33]:
def _condense_notes_per_csn(group: pd.DataFrame) -> str:
    """Helper to condense text notes for a patient_id group."""
    # For rows with the same time_difference, we want to combine them into a single entry
    group = group.groupby('relative_time_mins').agg({
        'Text': lambda x: ";".join(x.astype(str))
    }).reset_index()

    return "\n".join(
        f"{row['relative_time_mins']} - {row['Text']}"
        for _, row in group.astype({'Text': str, 'relative_time_mins': str}).iterrows()
    )
# group by patient_id and relative_time_mins, and then condense the text notes
condensed_df = merged_df.groupby('patient_id').apply(_condense_notes_per_csn).reset_index()
condensed_df.rename(columns={0: 'Text'}, inplace=True)
condensed_df.head()

/tmp/ipykernel_1841317/3955256315.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  condensed_df = merged_df.groupby('patient_id').apply(_condense_notes_per_csn).reset_index()


,patient_id,Text
0,115967095,0.0 - Glucose SerPl-mCnc: 120.0;Eosinophil: 0....
1,115967096,0.0 - Resp rate: 18.0;Body height: 64.17\n66.0...
2,115967097,0.0 - Transpl surg Note
3,115967098,0.0 - White blood cell count: 19.0;Red blood c...
4,115967100,0.0 - Blood potassium measurement: 4.0;Ca-I Se...


In [34]:
demographics = pd.read_parquet('hypoglycemia_demo.parquet')
# keep only the patient_id and age columns
demographics = demographics[['patient_id', 'gender', 'age']]
# demo_text = 'Gender: ' + gender + ', Age: ' + age
demographics['DemoText'] = demographics.apply(lambda x: 'Gender' + ': ' + str(x['gender']) + ', ' + 'Age' + ': ' + str(x['age']), axis=1)
demographics = demographics[['patient_id', 'DemoText']]

In [35]:
# merge condensed_df and demographics on patient_id
# if patient_id is not in condensed_df, then do not include it in the merged_df
condensed_df = condensed_df.merge(demographics, on='patient_id', how='inner')

In [36]:
condensed_df.head()

,patient_id,Text,DemoText
0,115967095,0.0 - Glucose SerPl-mCnc: 120.0;Eosinophil: 0....,"Gender: F, Age: 74"
1,115967096,0.0 - Resp rate: 18.0;Body height: 64.17\n66.0...,"Gender: F, Age: 68"
2,115967097,0.0 - Transpl surg Note,"Gender: F, Age: 49"
3,115967098,0.0 - White blood cell count: 19.0;Red blood c...,"Gender: F, Age: 53"
4,115967100,0.0 - Blood potassium measurement: 4.0;Ca-I Se...,"Gender: F, Age: 32"


In [37]:
# now concatenate the DemoText and Text columns into a single column called 'Text'
condensed_df['Text'] = condensed_df['DemoText'] + '\n' + condensed_df['Text']

In [38]:
# now merge condensed_df with labels on patient_id
condensed_df = condensed_df.merge(labels[['patient_id', 'label']], on='patient_id', how='inner')

In [40]:
condensed_df.head()

,patient_id,Text,DemoText,label
0,115967095,"Gender: F, Age: 74\n0.0 - Glucose SerPl-mCnc: ...","Gender: F, Age: 74",False
1,115967096,"Gender: F, Age: 68\n0.0 - Resp rate: 18.0;Body...","Gender: F, Age: 68",False
2,115967097,"Gender: F, Age: 49\n0.0 - Transpl surg Note","Gender: F, Age: 49",False
3,115967098,"Gender: F, Age: 53\n0.0 - White blood cell cou...","Gender: F, Age: 53",False
4,115967100,"Gender: F, Age: 32\n0.0 - Blood potassium meas...","Gender: F, Age: 32",False


In [41]:
# rename label to Label
condensed_df.rename(columns={"label": "Label"}, inplace=True)

In [43]:
# save to llm_full_hypoglycemia.parquet
condensed_df.to_parquet('textified_hypoglycemia_full.parquet', index=False)

In [44]:
# also, split the data into val and test sets, in a 70-30 split
from sklearn.model_selection import train_test_split
val_df, test_df = train_test_split(condensed_df, test_size=0.3, random_state=1234)
val_df.to_parquet('textified_hypoglycemia_val.parquet', index=False)
test_df.to_parquet('textified_hypoglycemia_test.parquet', index=False)